# Qwen3.5-0.8B Continued Pre-Training (CPT) Pipeline
### Kaggle TPU v5e-8 Acceleration Notebook (8 Pod Cores — Native BF16)

This notebook runs full-parameter Continued Pre-Training on **Qwen3.5-0.8B-Base** (~1B target tokens) utilizing **Google TPU v5e-8** (8 tensor cores, 128 GB total HBM):

| Feature | TPU v5e-8 Setup |
|:---|:---|
| **Compute Architecture** | 8 TPU v5e Cores (~1,576 TFLOPS BF16 Peak) |
| **Memory** | 128 GB Total HBM (16 GB per core) |
| **Precision** | Native Hardware `bfloat16` (`bf16=True`) |
| **Estimated Time (1B Tokens)** | **~2 to 4 Hours Total** (Completes in a single Kaggle session!) |
| **Sequence Length** | 2048 (Packed sequences, uniform static shape) |

**Dataset mixture:**
- 35% Stack v3 Code (`HuggingFaceCode/stack-v3-train`)
- 20% Stack v3 Documentation (`.md`, `.rst`, `README`)
- 20% The Vault (`Fsoft-AIC/the-vault-function`)
- 15% FineWeb-HQ (`epfml/FineWeb-HQ`)
- 10% OpenWebMath (`open-web-math/open-web-math`)

## 0. Directory Setup & Repository Working Path

In [ ]:
import os, sys
# Ensure working directory is set to QaptaanLM-0.75B
if os.path.exists("/kaggle/working/QaptaanLM-0.75B"):
    os.chdir("/kaggle/working/QaptaanLM-0.75B")
elif os.path.exists("QaptaanLM-0.75B"):
    os.chdir("QaptaanLM-0.75B")
print(f"✓ Current working directory: {os.getcwd()}")
sys.path.insert(0, os.getcwd())

## 1. Verify TPU v5e-8 Hardware & PyTorch/XLA Cores

In [ ]:
import torch
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
    print(f"✓ PyTorch/XLA version: {torch.__version__}")
    devices = xm.get_xla_supported_devices()
    print(f"✓ Detected XLA devices: {devices}")
    print(f"✓ TPU Device Count: {len(devices) if devices else 'XLA Active'}")
except ImportError as e:
    print(f"⚠ torch_xla not installed directly: {e}")
    print("Make sure 'TPU v5e-8' is selected in Kaggle's Settings > Accelerator")

## 2. Install Dependencies (Hugging Face + Accelerate)

In [ ]:
# Install core packages. Note: bitsandbytes and liger-kernel are CUDA-specific and omitted for TPU.
!pip install -q --upgrade transformers datasets accelerate peft \
    datasketch xxhash pyyaml rich huggingface_hub

## 3. Hugging Face Authentication

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✓ Logged in to Hugging Face")
except Exception as e:
    print(f"Manual login needed or secret not found: {e}")

## 4. Prepare Training Shards

In [ ]:
!mkdir -p /kaggle/working/data
!mkdir -p /kaggle/working/checkpoints
!mkdir -p /kaggle/working/logs

# If shards already exist in /kaggle/input/, skip generation. Otherwise run:
# !python scripts/03_process_data.py --output-dir /kaggle/working/data

## 5. Launch CPT Training on TPU v5e-8

We use `PJRT_DEVICE=TPU python scripts/05_train_cpt.py` to execute directly on the TPU PJRT runtime in native `bfloat16` without multi-process port collisions.

In [ ]:
# Launch training with PyTorch/XLA PJRT runtime on TPU v5e-8
# Direct execution avoids SliceBuilder port 8471 collisions and deadlocks
!PJRT_DEVICE=TPU python scripts/05_train_cpt.py --data-dir /kaggle/working/data


## 6. Evaluate Model & Compare Baselines

In [ ]:
!python scripts/06_evaluate.py --compare --base Qwen/Qwen3.5-0.8B-Base --cpt /kaggle/working/checkpoints/final --output /kaggle/working/logs/comparison.json